# 03 — Silver MERGE Upsert

**Project:** Orchestrated Incremental Orders Pipeline with Databricks Workflows  
**Layer:** Silver  
**Source:** `orders_workflow_project.bronze_orders_raw`  
**Target:** `orders_workflow_project.silver_orders_current`

This notebook processes one Bronze batch at a time and updates the Silver current-state table using Delta Lake `MERGE INTO`.

The notebook is designed to be executed by a Databricks Workflow / Job using parameters:

- `pipeline_run_id`
- `batch_id`

The Silver layer keeps only the latest known state of each order.

This notebook demonstrates:

- Incremental processing
- Delta Lake MERGE INTO
- Upserts
- Deduplication by order_id
- Current-state table maintenance
- Audit logging
- Protection against older events overwriting newer records

In [0]:
dbutils.widgets.text("pipeline_run_id", "manual_run", "Pipeline Run ID")

dbutils.widgets.dropdown(
    "batch_id",
    "1",
    ["1", "2", "3"],
    "Batch ID"
)

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
batch_id = int(dbutils.widgets.get("batch_id"))

print(f"Pipeline Run ID: {pipeline_run_id}")
print(f"Selected Batch ID: {batch_id}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window

In [0]:
spark.sql("USE SCHEMA orders_workflow_project")

In [0]:
bronze_table = "orders_workflow_project.bronze_orders_raw"
silver_table = "orders_workflow_project.silver_orders_current"
audit_table = "orders_workflow_project.pipeline_run_audit"

print(f"Bronze table: {bronze_table}")
print(f"Silver table: {silver_table}")
print(f"Audit table: {audit_table}")

In [0]:
bronze_batch_df = (
    spark.table(bronze_table)
    .filter(col("batch_id") == batch_id)
)

bronze_batch_count = bronze_batch_df.count()

print(f"Bronze records found for batch {batch_id}: {bronze_batch_count}")

display(
    bronze_batch_df.orderBy("order_id", "event_ts")
)

In [0]:
if bronze_batch_count == 0:
    raise Exception(
        f"No records found in Bronze for batch_id {batch_id}. "
        f"Run Bronze ingestion for this batch before running Silver."
    )

In [0]:
dedup_window = Window.partitionBy("order_id").orderBy(col("event_ts").desc())

silver_updates_df = (
    bronze_batch_df
    .withColumn("row_num", row_number().over(dedup_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_amount"),
        col("event_ts").alias("last_event_ts"),
        col("batch_id").alias("last_batch_id"),
        col("source_system"),
        col("source_batch_table"),
        col("bronze_ingestion_ts")
    )
    .withColumn("silver_processed_ts", current_timestamp())
)

silver_updates_count = silver_updates_df.count()

print(f"Silver updates prepared: {silver_updates_count}")

display(
    silver_updates_df.orderBy("order_id")
)

In [0]:
silver_updates_df.createOrReplaceTempView("silver_updates_temp")

print("Temporary view created: silver_updates_temp")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

print(f"Silver table exists: {silver_exists}")

In [0]:
if not silver_exists:
    if batch_id != 1:
        raise Exception(
            f"Silver table does not exist. "
            f"The first Silver execution must process batch_id = 1, but received batch_id = {batch_id}."
        )

    silver_updates_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(silver_table)

    task_status = "SUCCESS"
    records_processed = silver_updates_count
    task_message = f"Silver table created from batch {batch_id}."

    print(task_message)

else:
    task_status = "PENDING_MERGE"
    records_processed = silver_updates_count
    task_message = f"Silver table exists. Ready to merge batch {batch_id}."

    print(task_message)

In [0]:
if silver_exists:
    spark.sql(f"""
        MERGE INTO {silver_table} AS target
        USING silver_updates_temp AS source
        ON target.order_id = source.order_id

        WHEN MATCHED AND source.last_event_ts > target.last_event_ts THEN
          UPDATE SET
            target.customer_id = source.customer_id,
            target.order_status = source.order_status,
            target.order_amount = source.order_amount,
            target.last_event_ts = source.last_event_ts,
            target.last_batch_id = source.last_batch_id,
            target.source_system = source.source_system,
            target.source_batch_table = source.source_batch_table,
            target.bronze_ingestion_ts = source.bronze_ingestion_ts,
            target.silver_processed_ts = source.silver_processed_ts

        WHEN NOT MATCHED THEN
          INSERT (
            order_id,
            customer_id,
            order_status,
            order_amount,
            last_event_ts,
            last_batch_id,
            source_system,
            source_batch_table,
            bronze_ingestion_ts,
            silver_processed_ts
          )
          VALUES (
            source.order_id,
            source.customer_id,
            source.order_status,
            source.order_amount,
            source.last_event_ts,
            source.last_batch_id,
            source.source_system,
            source.source_batch_table,
            source.bronze_ingestion_ts,
            source.silver_processed_ts
          )
    """)

    task_status = "SUCCESS"
    task_message = f"MERGE completed for batch {batch_id}."

    print(task_message)

else:
    print("MERGE skipped because Silver table was created for the first time.")

In [0]:
pipeline_run_id_sql = pipeline_run_id.replace("'", "''")
task_name_sql = f"03_silver_merge_upsert_batch_{batch_id}"
status_sql = task_status.replace("'", "''")
message_sql = task_message.replace("'", "''")

spark.sql(f"""
INSERT INTO {audit_table}
SELECT
    '{pipeline_run_id_sql}' AS pipeline_run_id,
    '{task_name_sql}' AS task_name,
    '{status_sql}' AS status,
    CAST({int(records_processed)} AS BIGINT) AS records_processed,
    '{message_sql}' AS message,
    current_timestamp() AS processed_ts
""")

print("Audit record inserted successfully.")

In [0]:
display(
    spark.sql("""
    SELECT
        order_id,
        customer_id,
        order_status,
        order_amount,
        last_event_ts,
        last_batch_id,
        silver_processed_ts
    FROM orders_workflow_project.silver_orders_current
    ORDER BY order_id
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        COUNT(*) AS total_current_orders,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM orders_workflow_project.silver_orders_current
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        b.order_id,
        COUNT(*) AS total_events_in_bronze,
        s.order_status AS current_status_in_silver,
        s.last_event_ts,
        s.last_batch_id
    FROM orders_workflow_project.bronze_orders_raw b
    LEFT JOIN orders_workflow_project.silver_orders_current s
        ON b.order_id = s.order_id
    GROUP BY
        b.order_id,
        s.order_status,
        s.last_event_ts,
        s.last_batch_id
    ORDER BY b.order_id
    """)
)

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY orders_workflow_project.silver_orders_current
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.pipeline_run_audit
    ORDER BY processed_ts DESC
    """)
)